# 04 — Results & Evaluation

Full-period OOS performance and equity curves.

In [ ]:
import sys, yaml, pickle
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

sys.path.insert(0, str(Path('..')))
from src.features import generate_features
from src.walk_forward import eva_full_result
from src.signals import build_position_holdN

with open('../configs/strategy.yaml') as f:
    cfg = yaml.safe_load(f)

HORIZON_BARS = cfg['horizon_bars']
FEATURE_COLS = cfg['features']

df_raw = pd.read_csv('../data/ETHUSDT.csv', parse_dates=['timestamp'], index_col='timestamp')
df_features, target = generate_features(df_raw, feature_cols=FEATURE_COLS, HORIZON_BARS=HORIZON_BARS)

with open('../data/wf_results.pkl', 'rb') as f:
    saved = pickle.load(f)

X_total = saved['X_total']
result  = saved['result']
print('Loaded walk-forward results.')

### Full Out-of-Sample Performance

In [ ]:
eva_full_result(df_features, X_total, result, HORIZON_BARS=HORIZON_BARS)

### Equity Curves vs Buy & Hold

In [ ]:
X_all    = pd.concat(X_total, axis=0)
r1       = df_features.loc[X_all.index, 'return_1'].astype(float)
bh_curve = (1 + r1).cumprod()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=bh_curve.index, y=bh_curve.values,
    name='Buy & Hold', line=dict(color='gray', dash='dash')
))

colors = {'LightGBM': '#2196F3', 'RandomForest': '#4CAF50', 'XGBoost': '#FF9800'}
for name in result:
    full_signals = pd.concat(result[name]['signals'])
    position     = build_position_holdN(full_signals, HORIZON_BARS=HORIZON_BARS)
    strategy_r   = (position * r1).dropna()
    equity       = (1 + strategy_r).cumprod()
    fig.add_trace(go.Scatter(
        x=equity.index, y=equity.values,
        name=name, line=dict(color=colors.get(name))
    ))

fig.update_layout(
    title='Equity Curve: Strategy vs Buy & Hold',
    xaxis_title='Date', yaxis_title='Cumulative Return',
    yaxis_type='log', hovermode='x unified', template='plotly_dark'
)
fig.show()